# Hồ sơ kỹ thuật cho mục 3.3.3 — Đối chiếu phương án làm sạch điện

## 1. Chạy lại phương án trong tài liệu


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"../data/26_2521001040_Triệu Hoàng Lan Anh_CSV1.csv",
    sep=";",
    dtype=str,
    keep_default_na=False
)

num_cols = [
    'Global_active_power', 'Global_reactive_power', 'Voltage',
    'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
]

MISSING_STRS = ['?', 'null', 'NULL', 'None', 'N/A', '-', '']
gio_tho_b1 = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
df.replace(MISSING_STRS, np.nan, inplace=True)

for col in num_cols:
    df[col] = df[col].astype(str).str.replace(r'[^\d,.\-]', '', regex=True)
    df[col] = df[col].str.replace(',', '.', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')

PHYSICAL_LIMITS = {
    'Global_active_power':   (0.0, 11.0),
    'Global_reactive_power': (0.0, 2.0),
    'Voltage':               (210.0, 255.0),
    'Global_intensity':      (0.0, 50.0),
    'Sub_metering_1':        (0.0, 90.0),
    'Sub_metering_2':        (0.0, 82.0),
    'Sub_metering_3':        (0.0, 35.0),
}

for col, (lower_bound, upper_bound) in PHYSICAL_LIMITS.items():
    df.loc[(df[col] < lower_bound) | (df[col] > upper_bound), col] = np.nan

full_date_range = pd.date_range(start='2006-12-16 17:24', end='2010-11-26 21:02', freq='1min')
assert len(df) == len(full_date_range)
doc_duoc_b1 = gio_tho_b1.notna()
assert (gio_tho_b1[doc_duoc_b1].to_numpy() == full_date_range[doc_duoc_b1]).all()
df.index = full_date_range

df[num_cols] = df[num_cols].interpolate(method='time')
df[num_cols] = df[num_cols].ffill().bfill()


## 2. So sánh với file điện sạch hiện có


In [2]:
power_da_co = pd.read_csv(r"../data/power_clean_csv1.csv", parse_dates=['datetime'])
assert len(df) == len(power_da_co), "Hai bản khác số dòng, chưa thể đối chiếu theo vị trí."
print("Số dòng:", len(df))
print("Nhãn thời gian trùng theo vị trí:", (df.index == power_da_co['datetime']).all())
ket_qua = []
for col in num_cols:
    lech = (df[col].to_numpy() - power_da_co[col].to_numpy())
    khop = np.isclose(df[col].to_numpy(), power_da_co[col].to_numpy(), rtol=1e-9, atol=1e-9, equal_nan=True)
    ket_qua.append({
        'cot': col,
        'so_dong_khac': int((~khop).sum()),
        'so_dong_khac_nguong_1e6': int((~np.isclose(df[col].to_numpy(), power_da_co[col].to_numpy(), rtol=1e-6, atol=1e-6, equal_nan=True)).sum()),
        'ty_le_khac_pct': (~khop).mean() * 100,
        'chenh_lech_lon_nhat': np.nanmax(np.abs(lech))
    })
doi_chieu_b1 = pd.DataFrame(ket_qua)
print(doi_chieu_b1.to_string(index=False))


Số dòng: 2075259
Nhãn thời gian trùng theo vị trí: True


                  cot  so_dong_khac  so_dong_khac_nguong_1e6  ty_le_khac_pct  chenh_lech_lon_nhat
  Global_active_power         57524                        0        2.771895         6.666667e-07
Global_reactive_power         24402                        0        1.175853         6.666667e-08
              Voltage        145593                        0        7.015654         1.507921e-05
     Global_intensity         45502                        0        2.192594         1.666667e-06
       Sub_metering_1         14668                        0        0.706803         4.285714e-06
       Sub_metering_2         34816                        0        1.677670         2.666667e-06
       Sub_metering_3         44636                        0        2.150864         1.821086e-06


## 3. Lưu bảng đối chiếu


In [3]:
doi_chieu_b1.to_csv(r"../data/doi_chieu_phuong_an_dien_B1.csv", index=False)
